# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is Ranking Signal Analysis, which will be presented in the form of a ranking problem. The objective is to rank the content pages according to their search performance, traffic data, click-through-rate, freshness, and search visibility in order to decide on the order of SEO review. The outcome of this problem will be an ordered list of pages that need to be reviewed with explanations as to why they should receive more attention.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Initially, the target is a provisional review priority score, derived from observed features such as decreasing trends, below-median click-through rate, weak average search rank, and old content. It represents an explicit proxy target rather than an implicit ground-truth label for the need for a page to be optimized. In the second step, labels can be collected by the work of SEO specialists.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import subprocess
import pandas as pd

repo_path = Path("/content/Flyrank-Ml-Pipeline")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline.git",
         str(repo_path)],
        check=True
    )

csv_path = repo_path / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

# Build a provisional, transparent review-priority proxy
ctr_median = df["ctr"].median()
update_days_median = df["days_since_last_update"].median()

df["review_priority_proxy_score"] = (
    2 * df["trend_direction"].eq("down").astype(int)
    + df["ctr"].le(ctr_median).fillna(False).astype(int)
    + df["avg_position"].gt(10).fillna(False).astype(int)
    + df["days_since_last_update"].gt(update_days_median)
        .fillna(False).astype(int)
)

print("Rows:", len(df))
print("Median CTR:", round(ctr_median * 100, 3), "%")
print("Median days since update:", round(update_days_median, 1))
print("Proxy score distribution:")
print(df["review_priority_proxy_score"].value_counts().sort_index())


Rows: 30000
Median CTR: 7.0 %
Median days since update: 20.0
Proxy score distribution:
review_priority_proxy_score
0    1859
1    5595
2    6273
3    8219
4    5374
5    2680
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The main success measure is the Precision at 10: the percentage of top 10 recommended pages which are recognized to need review by SEO experts. This list of ranking must be better than some naive baseline like ranking pages by descending trend only. Because there is no information about whether these pages really need review, this measure cannot be computed yet.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

ranked_preview = df.sort_values(
    "review_priority_proxy_score",
    ascending=False
)

print("Pages available for ranking:", len(ranked_preview))
print("Top 10 proxy scores:")
print(ranked_preview["review_priority_proxy_score"].head(10).tolist())


Pages available for ranking: 30000
Top 10 proxy scores:
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Analysis unit refers to one single content record corresponding to one page or content object. One row holds the SEO performance indicators and review priority score. One and the same page can possess various measured attributes, but still each row represents one option for prioritization.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

columns_to_show = [
    "content_id",
    "trend_direction",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "review_priority_proxy_score"
]

display(
    df[columns_to_show]
    .sort_values("review_priority_proxy_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)


,content_id,trend_direction,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,review_priority_proxy_score
0,content_91067a14431a,down,179,0,0.00,27.1,104,5
1,content_4433db877b6a,down,116,0,0.00,14.9,28,5
2,content_f146d105b2be,down,7116,4,0.06,29.6,104,5
3,content_cd6760921db8,down,988,0,0.00,47.3,41,5
4,content_049e9c466e09,down,208,0,0.00,37.0,104,5
5,content_1cc38b3f04d2,down,9,0,0.00,49.7,92,5
6,content_a0404909ecca,down,1115,0,0.00,14.1,104,5
7,content_3170de7e27d9,down,167,0,0.00,40.6,28,5
8,content_fe3f8dbf3614,down,277,0,0.00,10.1,104,5
9,content_aea3a584f264,down,537,0,0.00,25.5,104,5


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Fixed criteria make for a good starting point, but SEO success involves a number of interacting signals. An SEO page that is experiencing low traffic and high visibility requires an intervention distinct from that of a dormant page with low impressions. With human-approved signal criteria, ML would be able to learn patterns of signals and enhance prioritization. I will be comparing any approach learned by the machine to a simple baseline criterion, assuming that ML will be superior before the evaluation is done.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Trend distribution:")
display(df["trend_direction"].value_counts(dropna=False))

print("Content freshness distribution:")
display(df["freshness_tier"].value_counts(dropna=False))

print("Summary of key ranking signals:")
display(
    df[[
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]].describe().round(3)
)


Trend distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


Content freshness distribution:


,count
freshness_tier,
0-30,20480
91-180,9171
31-90,175
181+,174


Summary of key ranking signals:


,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update
count,30000.000,30000.000,30000.000,30000.000,30000.000
mean,5200.366,16.097,0.511,16.342,46.098
std,16838.020,75.077,3.279,15.217,42.079
min,1.000,0.000,0.000,0.000,1.000
25%,81.000,0.000,0.000,6.200,20.000
50%,731.000,1.000,0.070,10.800,20.000
75%,3615.250,7.000,0.290,22.300,104.000
max,517715.000,4178.000,100.000,245.000,373.000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.